## ▶ What you'll see when you run this
- Keyword search **misses** a sentence that clearly means the same thing, then embeddings + cosine similarity **find it** — and a heatmap shows why.

**Time:** ~10 min · **Cost:** free (local Hugging Face model, no LLM call) · **Keys:** none

# Week 10 · Notebook 0 — Embeddings Intuition (build it before the database)
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Before you trust a vector database, build the idea yourself:
1. Watch **keyword search fail** on a meaning-based question (the dumb baseline).
2. Turn sentences into **embeddings** with a free local model.
3. Compute **cosine similarity by hand** with NumPy and **see it as a heatmap**.
4. Write **top-k semantic search from scratch** (it's ~5 lines).
5. Only THEN meet **ChromaDB** — the scaled-up version of what you just built.

> Runs in Google Colab. No API key, no OpenAI — embeddings are local Hugging Face.

## 0. Install
`sentence-transformers` gives us a small, free embedding model; `matplotlib` draws the heatmap; `chromadb` is only needed for the very last section.

In [ ]:
!pip -q install sentence-transformers matplotlib numpy chromadb

## 1. Our tiny corpus
Ten short sentences on a few themes (pets, weather, finance, programming). We'll search them two ways and compare.

In [ ]:
corpus = [
    'The cat napped on the warm windowsill.',
    'A kitten chased a ball of yarn across the room.',
    'My dog loves long walks in the park.',
    'It rained heavily all afternoon and the streets flooded.',
    'A gentle breeze and clear skies made for a lovely day.',
    'The stock market dropped sharply after the report.',
    'Investors worried about rising interest rates.',
    'Python is a popular language for data science.',
    'I debugged the function until all the tests passed.',
    'The recipe needs two cups of flour and one egg.',
]
for i, s in enumerate(corpus):
    print(i, s)

## 2. The dumbest baseline: keyword / substring search
Ask for a **young cat**. A human knows sentence 1 ("cat") is a match. But our query words don't literally appear, so substring search returns **nothing useful**.

In [ ]:
def keyword_search(query, corpus):
    q_words = set(query.lower().split())
    hits = []
    for i, s in enumerate(corpus):
        s_words = set(s.lower().replace('.', '').split())
        overlap = q_words & s_words
        if overlap:
            hits.append((i, overlap))
    return hits

query = 'a young feline resting'
print('Query:', query)
print('Keyword hits:', keyword_search(query, corpus))
print('\n-> None of these words appear literally, so keyword search FINDS NOTHING,')
print('   even though sentences 0 and 1 are obviously about cats.')

## 3. Embeddings = meaning as a vector
A **sentence embedding** turns each sentence into a fixed-length list of numbers (here 384) that captures *meaning*. Sentences that mean similar things get similar vectors. We use a small, free, **local** Hugging Face model.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')   # 384-dim, free, local
embeddings = model.encode(corpus)
print('embeddings shape:', embeddings.shape)   # (10 sentences, 384 numbers each)
print('first sentence, first 8 numbers:', np.round(embeddings[0][:8], 3))

## 4. Cosine similarity, by hand
**Cosine similarity** measures the *angle* between two vectors, ignoring length:

$$\cos(\mathbf{a},\mathbf{b}) = \frac{\mathbf{a}\cdot\mathbf{b}}{\lVert\mathbf{a}\rVert\,\lVert\mathbf{b}\rVert}$$

It runs from -1 (opposite) to 1 (identical direction). We write it with NumPy — no library magic.

In [ ]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print('cat (0)    vs kitten (1):', round(cosine(embeddings[0], embeddings[1]), 3))
print('cat (0)    vs market (5):', round(cosine(embeddings[0], embeddings[5]), 3))
print('market (5) vs rates  (6):', round(cosine(embeddings[5], embeddings[6]), 3))
print('\nSame topic -> high score; different topic -> low score.')

## 5. The whole similarity matrix as a heatmap
Compare every sentence to every other sentence. Bright blocks along the diagonal reveal the **themes** the model discovered — pets, weather, finance, programming — with no labels given to it.

In [ ]:
import matplotlib.pyplot as plt

# Normalize rows so a single matrix multiply gives all pairwise cosines.
norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
sim = norm @ norm.T          # (10, 10) cosine-similarity matrix

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim, cmap='viridis')
ax.set_xticks(range(len(corpus)))
ax.set_yticks(range(len(corpus)))
ax.set_title('Sentence-to-sentence cosine similarity')
fig.colorbar(im, label='cosine similarity')
plt.tight_layout(); plt.show()

print('Bright off-diagonal cells = different sentences the model thinks are related.')

## 6. Top-k semantic search, from scratch
This is the whole trick behind a vector database, in a few lines: **embed the query, cosine-compare it to every stored vector, sort, take the top k.** Re-run the same query that keyword search whiffed on.

In [ ]:
def semantic_search(query, corpus, embeddings, k=3):
    q = model.encode([query])[0]
    scores = [cosine(q, e) for e in embeddings]       # compare to every sentence
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return ranked[:k]

query = 'a young feline resting'
print('Query:', query, '\n')
for idx, score in semantic_search(query, corpus, embeddings, k=3):
    print(f'  {score:.3f}  [{idx}] {corpus[idx]}')
print('\n-> Embeddings find the cat sentences that keyword search MISSED.')

## 7. Check yourself
Try a finance query and a programming query. Notice you never typed the matching words — the model matches on **meaning**.

In [ ]:
for q in ['worries about the economy', 'fixing code with tests']:
    print('Query:', q)
    for idx, score in semantic_search(q, corpus, embeddings, k=2):
        print(f'  {score:.3f}  [{idx}] {corpus[idx]}')
    print()

## 8. ChromaDB = the scaled version of what you just built
You wrote: store vectors → cosine-compare a query → sort → take top-k. A **vector database** does exactly that, but fast over millions of items, with persistence and metadata. The mental model is unchanged — `col.query(...)` is your `semantic_search(...)` with an index under the hood.

In [ ]:
import chromadb
client = chromadb.Client()                 # in-memory
col = client.create_collection('week10_intuition')
col.add(documents=corpus, ids=[str(i) for i in range(len(corpus))])

res = col.query(query_texts=['a young feline resting'], n_results=3)
for doc, dist in zip(res['documents'][0], res['distances'][0]):
    print(f'  dist={dist:.3f} | {doc}')

print('\nSame top results as your from-scratch search — now backed by an index.')
print('(Chroma returns a DISTANCE: smaller = closer. Your cosine was a SIMILARITY:')
print(' larger = closer. Same ordering, flipped scale.)')

## 9. Recap
- **Keyword search** matches letters; it misses meaning.
- An **embedding** is meaning as a vector; **cosine similarity** measures closeness.
- **Top-k semantic search** = embed query → cosine vs all → sort → take k.
- A **vector store (ChromaDB)** is that same idea, scaled and indexed.

Next: Notebook 1 wires this into **LangChain**, and Notebook 2 goes deeper on **ChromaDB**. You now know what they're doing underneath.